# Constraint Satisfaction
## Eight Queens Revisited
### Introduction
You are already familiar with the eight queens problem, now let's look at how we can solve it using *constraint satisfaction*.

### Constraint Satisfaction
We have already been thinking of the eight queens problem as trying to assign values to variables. In doing local search we moved pieces at random and checked whether we had improved the solution. Now, we will start with an empty board (or optionally one that is partially full), and then we will try placing pieces one by one in valid spaces. This is probably similar to how you would solve the problem by hand.

The key part, and the essence of the technique, is that for each variable (column) we keep track of its *domain*, its list of possible values (rows). If we decide to place a piece in the second column, we must update the domain of every other column to indicate which values are still permitted. If at some point we have no possible valid options for a column, we must backtrack and try another option. This is the combination of depth first search and the propagation of constraints.

### Eight Queens Code (Again)
This time we'll need to interact with the state at a deeper level, modifying the individual components, not just treating each state as a black box. The class below is a *partial* eight queens state because it can have any number of its columns allocated, while keeping track of the values that are still possible in the other columns.

Particularly take care when reading the `set_value` method, as this encapsulates the mechanics of the problem and propagates the constraints.

In [ ]:
import copy

# An object of this class represents one point in the search: some columns have queens placed, the rest don't yet.
# The class is the board. It stores where queens are and which squares are still safe, and it knows the rules. But it never decides anything; it just does what it's told.
class PartialEightQueensState:
    def __init__(self, n=8):
        self.n = n

        # A list of possible values for each column, possible_values holds only the domains, one per variable
        # The inner part [i for i in range(0, self.n)] makes [0, 1, 2, ..., 7], the list of rows a queen could go in. 
        # The outer part for _ in range(0, self.n) repeats that once per column, giving 8 separate lists.
        # So possible_values[3] is the list of rows still allowed for column 3. Each column gets its own independent list
        # Important: possible_values[col] doesn't mean "free squares left in this column". 
        # It means "rows where this column's queen could be". 
        # Once the queen is placed, the answer to that question is "exactly the row it's in". So the domain becomes [row], not empty.
        self.possible_values = [[i for i in range(0, self.n)] for _ in range(0, self.n)]
        # Makes [-1, -1, -1, -1, -1, -1, -1, -1]. This records which row each column has been assigned, with -1 meaning "not assigned yet":
        self.final_values = [-1] * self.n

    def is_goal(self):
        """
        Checker only: does not modify the state.
        Goal test - returns True if every column has been assigned a row
        (no -1 left in final_values), otherwise False.

        No need to check for attacking queens: 'set_value' only allows
        values still in the domain, so any complete assignment is valid.
        """

        return all(value != -1 for value in self.final_values)

    def is_invalid(self):
        """
        Checker only: does not modify the state.
        Dead-end test - returns True if any column's domain is empty
        (no safe row left for its queen), otherwise False.
        True means this branch cannot lead to a solution, so backtrack.
        """
        #Loops over every column's domain and checks whether it's empty. any(...) returns True if at least one is. 
        # An empty domain means some column has nowhere safe left for a queen, so this branch of the search is a dead end 
        # and we need to backtrack.
        return any(len(values) == 0 for values in self.possible_values)

    def get_possible_values(self, column):
        # hands back a duplicate list, so whoever calls this can change it without damaging the state. 
        # This matters because order_values later shuffles the list it receives.
        return self.possible_values[column].copy()

    def get_final_state(self):
        # If the board is complete, it returns the solution, e.g. [2, 4, 6, 0, 3, 1, 7, 5], meaning column 0's queen is in row 2, column 1's in row 4, and so on.
        # Otherwise it returns -1 to signal "no complete solution here":
        if self.is_goal():
            return self.final_values
        else:
            return -1

    def get_singleton_columns(self):
        """Returns the columns which have no final value but exactly 1 possible value"""
        # When you place a queen, set_value removes attacked rows from the other columns. Sometimes that pruning leaves a column with only one option
        # At the end of set_value, the code uses this list to place the forced queens automatically, instead of making the search "choose" something that isn't really a choice.
        # This saves the search from exploring branches that are already decided or already doomed.
        return [index for index, values in enumerate(self.possible_values)
                # only those with exactly one row left and not yet assigned. These are forced moves: there's only one place the queen can go, but we haven't put it there yet:
                if len(values) == 1 and self.final_values[index] == -1]

    def set_value(self, column, row):
        """Returns a new state with this column set to this row, and the change propagated to other domains"""
        # It places a queen at (column, row) and works out the consequences.

        #The below asks "is this row still in this column's domain?" If the row was removed earlier because another queen attacks it, the answer is no:
        if row not in self.possible_values[column]:
            raise ValueError(f"{row} is not a valid choice for column {column}")

        # create a deep copy: the method returns a new state, does not modify the existing one
        # This is what makes backtracking easy: if this choice turns out badly, 
        # the search just throws the copy away and still has the original to try a different value from
        state = copy.deepcopy(self)

        # These two lines do the same thing to two different lists, because the state keeps track of two different things, and both need updating when a queen is placed.
        # See next cell markdown for an example walkthrough
        state.possible_values[column] = [row]
        state.final_values[column] = row

        # now update all other columns possible values
        # Loops over every column to the left of the new queen. If column is 3, update_col takes values 0, 1, 2.
        for update_col in range(0, column):
            # remove same row for all columns to the left fo the queen, one at a time:
            if row in state.possible_values[update_col]:
                state.possible_values[update_col].remove(row)

            # remove upper diagonal
            # A diagonal moves one row per column, so the attacked square is that same distance away vertically.
            # Note: The diagonal formula sometimes produces a row number that doesn't exist on the board, and the if check deals with that without any extra code.
            upper_diag = row + (column - update_col)
            if upper_diag in state.possible_values[update_col]:
                state.possible_values[update_col].remove(upper_diag)

            # lower diagonal
            # Any negative rows are never in a domain, so they're ignored automatically.
            lower_diag = row - (column - update_col)
            if lower_diag in state.possible_values[update_col]:
                state.possible_values[update_col].remove(lower_diag)

        # now update columns to the right, same as above just the columns to the right now looking at same row, upper and lower diagonal:
        for update_col in range(column + 1, state.n):
            # remove same row
            if row in state.possible_values[update_col]:
                state.possible_values[update_col].remove(row)

            # remove upper diagonal
            upper_diag = row + (update_col - column)
            if upper_diag in state.possible_values[update_col]:
                state.possible_values[update_col].remove(upper_diag)

            # lower diagonal
            lower_diag = row - (update_col - column)
            if lower_diag in state.possible_values[update_col]:
                state.possible_values[update_col].remove(lower_diag)

        # By the time this code runs, set_value has already:
        # 1. Checked the chosen row is allowed (the ValueError guard)
        # 2. Made a copy of the state (deepcopy)
        # 3. Placed the queen (updated possible_values[column] and final_values[column])
        # 4. Pruned every other column's domain (same row plus both diagonals)
        # Step 4 can leave some columns with only one safe row. This last block finds those forced moves and makes them straight away.
        # if any other columns with no final value only have 1 possible value, make them final
        singleton_columns = state.get_singleton_columns()
        while len(singleton_columns) > 0:
            # Takes the first forced column as singleton_columns is a list of column numbers that are forced.
            col = singleton_columns[0]
            # Note possible_values is one list of allowed rows per column. The position in the outer list is the column number.
            # In general, a column's domain in possible_values can hold several rows, like [0, 2, 3]. But get_singleton_columns only returns columns whose domain has exactly one row.
            # [0] after [col] still needed as otherwise you would get a list back (with a single element) rather than just the element
            # Note this is a recursive call:
            # A forced queen is still a real queen. Once it's on the board, it attacks its row and diagonals too. So the other columns' domains have to be pruned for it, just like for the first queen.
            # Each set_value call returns as soon as its own while loop finds no forced columns left. The innermost call finishes first, then the one that called it, and so on back up.
            state = state.set_value(col, state.possible_values[col][0])
            # After placing a forced queen, it checks again from scratch for forced columns, and the result decides whether the loop runs another time.
            singleton_columns = state.get_singleton_columns()

        # Returns the new state, with the queen placed, all domains pruned, and all forced moves made. 
        # The caller then checks is_goal() or is_invalid() to decide what to do next.   
        return state

### Reminder: why `set_value` updates two lists

When a queen is placed, the state tracks two different things, so both need updating:

```python
state.possible_values[column] = [row]   # "this column has no other options now"
state.final_values[column] = row        # "this column is done"
```

**Example:** placing a queen in column 3, row 5 on an empty board

| | Before | After |
|---|---|---|
| `possible_values[3]` (domain: rows still allowed) | `[0, 1, 2, 3, 4, 5, 6, 7]` | `[5]` |
| `final_values[3]` (assignment: `-1` = no queen) | `-1` | `5` |

**Why both are needed:**
- `possible_values` is read by the search (to pick values) and by `is_invalid` (to spot empty domains). If it weren't shrunk, the search could try to place a second queen in column 3.
- `final_values` is read by `is_goal` (to check every column is assigned) and holds the final answer.

**If only one was updated:**
- Only `final_values` → the domain still shows 8 options, so it's out of date.
- Only `possible_values` → the column looks like a singleton (1 value but still `-1`), so `get_singleton_columns` would try to place the queen again.

### Depth First Search with Constraint Propagation
Now finding a solution is a simple matter of picking a column to set, picking a value to set to that column, finding the resulting state with constraint propagation, and then searching all possible options until we find a solution.

Here is the pseudocode from Russell and Norvig (p. 215):

<br>
<figure>
<img src="resources/constraint_propagation.png", width=600>
</figure>
<br>


In [11]:
# This is the search part: the BACKTRACK function from the Russell and Norvig pseudocode
# this code decides which choices to try, and in what order.
# The functions are the player using the board above. depth_first_search (with its two helpers) decides which move to try, asks the board to make it, looks at the result, and decides what to do next.
import random


# Important: possible_values[col] doesn't mean "free squares left in this column". 
# It means "rows where this column's queen could be". 
# Once the queen is placed, the answer to that question is "exactly the row it's in". So the domain becomes [row], not empty.


def pick_next_column(partial_state):
    # pick_next_column answers one question: "which empty column should I place a queen in next?"
    """
    Used in depth first search, currently chooses a random 
    column that has more than one possible value

    This is SELECT-UNASSIGNED-VARIABLE in the pseudocode.
    """
    # Goes through each column and keeps the column number if its domain has more than one row:
    col_indices = [index for index, values in enumerate(partial_state.possible_values) if len(values) > 1]
    return random.choice(col_indices)


def order_values(partial_state, col_index):
    """
    Get values for a particular column in the 
    order we should try them in. Currently random.
    """
    values = partial_state.get_possible_values(col_index)
    random.shuffle(values)
    return values


def depth_first_search(partial_state=PartialEightQueensState()): # 1. the board at this point
    """
    This will do a depth first search on partial states, trying 
    each possible value for a single column.

    Notice that we do not need to try every column: if we try 
    every possible value for a column and can't find a
    solution, then there is no possible value for this column, 
    so there is no solution.
    """
    col_index = pick_next_column(partial_state)                  # 2. the column being chosen
    values = order_values(partial_state, col_index)

    for value in values:                                         # 3. where it is in the list of rows
        new_state = partial_state.set_value(col_index, value)
        if new_state.is_goal():
            return new_state
        if not new_state.is_invalid():  
            deep_state = depth_first_search(new_state)          # go one choice deeper
            if deep_state is not None and deep_state.is_goal():
                return deep_state
    return None


partial_state = PartialEightQueensState(n=8)
goal = depth_first_search(partial_state).get_final_state()
print(goal)

[2, 4, 6, 0, 3, 1, 7, 5]


This is the most basic form of the technique, but it is quite powerful. It can easily handle quite large board sizes, much larger than we could do using local search. Try some out by changing the value of `n` above.

Now consider a few ways to improve this code:
* Add some metrics to calculate how many partial states are generated to better compare with other methods
* Modify the code so that you can pass an initial configuration into the `PartialEightQueensState` constructor, which should then run these values through `set_value` to update possible values
 * Can you find a starting state which has no possible solution, but does not start with two queens attacking each other?
* Change how the algorithm picks the next column. It can be more efficient to pick the column which currently has the fewest options. Compare your metric from before.
* Change how the algorithm picks which order to try the values it assigns to a column. One idea might be to prioritise values which *least constrain* other columns, but watch out this doesn't end up adding more complexity than it saves.
 * One way to check this would be to try timing your code. You can use `time.time()` or the [`timeit` module](https://docs.python.org/3.8/library/timeit.html).

### Metrics: counting partial states

In the local search notebook the shared metric was `states_generated`: every board built and costed. The equivalent here is every `PartialEightQueensState` built by `set_value`, because each call does one `deepcopy` and one round of constraint propagation. That *is* the work, so that's what we count.

Two things make counting here a bit different from hill climbing:

1. `set_value` calls itself for forced (singleton) columns, so one call from the search can quietly build several states. We recover that number by comparing how many columns were assigned before and after the call: each recursive `set_value` places exactly one queen, so `after - before` is the number of states built.
2. The search is recursive, so a `SearchMetrics` object is passed down and every level adds to the same one, rather than trying to return counts back up.

The board class is left untouched: it still never decides anything, and it doesn't count anything either. The counting lives with the player (the search).

| Metric | What it counts |
|---|---|
| `states_generated` | every partial state built (`choices + forced`) - **the shared metric** |
| `choices` | `set_value` calls made by the search itself, one per value tried |
| `forced` | queens placed automatically by singleton propagation inside `set_value` |
| `dead_ends` | generated states with an empty domain (`is_invalid`) |
| `backtracks` | times a column ran out of values and the search gave up on it |
| `nodes_expanded` | `depth_first_search` calls, i.e. columns picked |
| `max_depth` | deepest recursion level reached (columns *chosen*, not queens on the board) |


In [12]:
class SearchMetrics:
    """Counters for one depth_first_search run. Passed down the recursion so every level updates the same object."""
    def __init__(self):
        self.states_generated = 0   # every partial state built by set_value (choices + forced) - the shared metric
        self.choices = 0            # set_value calls made by the search itself (one per value tried)
        self.forced = 0             # queens placed automatically by singleton propagation inside set_value
        self.dead_ends = 0          # generated states with an empty domain (is_invalid)
        self.backtracks = 0         # times a column ran out of values and the search gave up on it
        self.nodes_expanded = 0     # depth_first_search calls, i.e. columns picked
        self.max_depth = 0          # deepest recursion level reached (columns chosen by the search, not queens on the board)

    def __repr__(self):
        return (f"states_generated={self.states_generated}, choices={self.choices}, forced={self.forced}, "
                f"dead_ends={self.dead_ends}, backtracks={self.backtracks}, "
                f"nodes_expanded={self.nodes_expanded}, max_depth={self.max_depth}")


def assigned_count(partial_state):
    """How many columns already have a queen (final_values != -1)."""
    return sum(1 for value in partial_state.final_values if value != -1)


def depth_first_search_metrics(partial_state=None, metrics=None, depth=0):
    """
    Same search as depth_first_search above, plus counting.
    :returns (goal state or None, SearchMetrics)
    """
    if partial_state is None:               # None default, not PartialEightQueensState(): see the mutable-default warning in the local search notebook
        partial_state = PartialEightQueensState()
    if metrics is None:                     # top-level call creates the counters; recursive calls receive them
        metrics = SearchMetrics()

    metrics.nodes_expanded += 1
    metrics.max_depth = max(metrics.max_depth, depth)

    col_index = pick_next_column(partial_state)
    values = order_values(partial_state, col_index)

    for value in values:
        before = assigned_count(partial_state)
        new_state = partial_state.set_value(col_index, value)
        after = assigned_count(new_state)

        # one deepcopy for our choice, plus one per forced queen placed inside set_value
        metrics.choices += 1
        metrics.forced += after - before - 1
        metrics.states_generated += after - before

        if new_state.is_goal():
            return new_state, metrics
        if new_state.is_invalid():
            metrics.dead_ends += 1              # this branch is dead, try the next value
            continue
        deep_state, _ = depth_first_search_metrics(new_state, metrics, depth + 1)
        if deep_state is not None:              # only ever non-None when it is a goal
            return deep_state, metrics

    metrics.backtracks += 1                     # every value for this column failed: give up on it, caller tries its next value
    return None, metrics


random.seed(0)                                  # this notebook uses random, not numpy, so seed random
goal, metrics = depth_first_search_metrics(PartialEightQueensState(n=8))
print(goal.get_final_state())
print(metrics)


[2, 4, 6, 0, 3, 1, 7, 5]
states_generated=49, choices=24, forced=25, dead_ends=14, backtracks=6, nodes_expanded=10, max_depth=4


**Reading the numbers against local search.** `states_generated` is the same *kind* of number as in the hill climbing / annealing notebook, so the tables can be read side by side, but the per-state cost differs: a local search state is one full board and one `cost()` call, while a partial state here is a `deepcopy` plus a domain update for every other column. Fewer states does not automatically mean less time, which is why `time_s` is kept alongside.

Each trial below starts from the same empty board; what changes between trials is only the random column and value order, so the spread shows how much luck is involved in the ordering (the later bullets are about removing that luck).

In [16]:
# Same shape as the compare() in the local search notebook, so the summary tables can be read side by side.
import time
import pandas as pd

def run_csp(n):
    """One DFS + constraint propagation run on an empty n-queens board. Returns a dict of metrics."""
    t0 = time.perf_counter()
    goal, m = depth_first_search_metrics(PartialEightQueensState(n=n))
    return {
        "algorithm": "DFS + constraint propagation",
        "n": n,
        "solved": goal is not None,
        "states_generated": m.states_generated,   # the shared metric
        "choices": m.choices,
        "forced": m.forced,
        "dead_ends": m.dead_ends,
        "backtracks": m.backtracks,
        "max_depth": m.max_depth,
        "time_s": time.perf_counter() - t0,
    }

def compare_csp(sizes=(8, 10, 15, 20), trials=200, seed=0):
    """Run the search `trials` times per board size. Each trial differs only in the random column/value order."""
    random.seed(seed)
    return pd.DataFrame([run_csp(n) for n in sizes for _ in range(trials)])

raw_csp = compare_csp()

summary_csp = (
    raw_csp.groupby("n")
           .agg(
               runs           = ("solved",           "size"),
               success_rate   = ("solved",           "mean"),
               avg_states     = ("states_generated", "mean"),
               median_states  = ("states_generated", "median"),
               avg_choices    = ("choices",          "mean"),
               avg_forced     = ("forced",           "mean"),
               avg_dead_ends  = ("dead_ends",        "mean"),
               avg_backtracks = ("backtracks",       "mean"),
               avg_time_s     = ("time_s",           "mean"),
           )
)

summary_csp.style.format({
    "runs":           "{:d}",
    "success_rate":   "{:.0%}",
    "avg_states":     "{:,.1f}",
    "median_states":  "{:,.0f}",
    "avg_choices":    "{:,.1f}",
    "avg_forced":     "{:,.1f}",
    "avg_dead_ends":  "{:,.1f}",
    "avg_backtracks": "{:,.1f}",
    "avg_time_s":     "{:.4f}",
})


,runs,success_rate,avg_states,median_states,avg_choices,avg_forced,avg_dead_ends,avg_backtracks,avg_time_s
n,,,,,,,,,
8,200,100%,26.2,20,11.1,15.1,5.1,1.8,0.0002
10,200,100%,54.6,42,21.3,33.2,11.1,4.6,0.0005
15,200,100%,103.1,68,35.4,67.7,18.1,7.8,0.0014
20,200,100%,217.0,116,65.1,151.9,35.5,16.2,0.0034
